#### Step:1 Data Loading

In [1]:
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes",
    "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label", "difficulty_level"
]

In [2]:
import pandas as pd

train_df = pd.read_csv(r"C:\Users\Suhitha\Downloads\archive (3)\nsl-kdd\KDDTrain+.txt", names=columns)
test_df = pd.read_csv(r"C:\Users\Suhitha\Downloads\archive (3)\nsl-kdd\KDDTest+.txt", names=columns)

train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


#### Step 2: Exploratory Data Analysis (EDA)

In [3]:
print(train_df.shape)
print(train_df['label'].value_counts())
print(train_df.dtypes)
print(train_df.isnull().sum().sum())

(125973, 43)
label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64
duration                         int64
protocol_type                      str
service                            str
flag                               str
src_bytes                        int64
dst_bytes                        int64
land                             int64
wrong_fragment                   int64
urgent                           int64
hot                          

#### Step 3: Binary Label Creation

In [4]:
train_df['binary_label']=train_df['label'].apply(lambda x: 0 if x == 'normal' else 1)
test_df['binary_label']=test_df['label'].apply(lambda x: 0 if x == 'normal' else 1)

In [5]:
train_df[['label','binary_label']].head(10)

,label,binary_label
0,normal,0
1,normal,0
2,neptune,1
3,normal,0
4,normal,0
5,neptune,1
6,neptune,1
7,neptune,1
8,neptune,1
9,neptune,1


#### Step 4: Categorical Encoding (One-Hot Encoding)

In [6]:
train_df=pd.get_dummies(train_df,columns=['protocol_type','service','flag'])
test_df=pd.get_dummies(test_df,columns=['protocol_type','service','flag'])

In [7]:
train_df.shape

(125973, 125)

In [8]:
train_df.filter(like='protocol_type').head(10)

,protocol_type_icmp,protocol_type_tcp,protocol_type_udp
0,False,True,False
1,False,False,True
2,False,True,False
3,False,True,False
4,False,True,False
5,False,True,False
6,False,True,False
7,False,True,False
8,False,True,False
9,False,True,False


In [9]:
train_df[['src_bytes', 'dst_bytes']].describe()

,src_bytes,dst_bytes
count,1.259730e+05,1.259730e+05
mean,4.556674e+04,1.977911e+04
std,5.870331e+06,4.021269e+06
min,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00
50%,4.400000e+01,0.000000e+00
75%,2.760000e+02,5.160000e+02
max,1.379964e+09,1.309937e+09


#### Step 5: Feature Scaling (StandardScaler)

In [10]:
numeric_cols=train_df.select_dtypes(include=['int64','float64']).columns.tolist()
numeric_cols.remove('binary_label')
numeric_cols.remove('difficulty_level')

In [11]:
from sklearn.preprocessing import StandardScaler


In [12]:
scaler = StandardScaler()
train_df[numeric_cols]=scaler.fit_transform(train_df[numeric_cols])
test_df[numeric_cols]=scaler.transform(test_df[numeric_cols])
train_df[numeric_cols].describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
count,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,...,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05,1.259730e+05
mean,2.425388e-17,5.640437e-19,4.512349e-19,-1.579322e-18,-1.128087e-18,-2.481792e-18,-1.895187e-17,4.512349e-18,5.820931e-17,-7.896611e-19,...,1.579322e-17,-5.730684e-17,1.053634e-16,-2.295658e-17,3.181206e-17,3.158645e-17,1.737255e-17,-2.594601e-17,4.873337e-17,7.648432e-17
std,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,...,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00
min,-1.102492e-01,-7.762241e-03,-4.918644e-03,-1.408881e-02,-8.948642e-02,-7.735985e-03,-9.507567e-02,-2.702282e-02,-8.092618e-01,-1.166364e-02,...,-1.836071e+00,-1.044721e+00,-1.161030e+00,-4.390782e-01,-4.801968e-01,-2.891034e-01,-6.395319e-01,-6.248708e-01,-3.876346e-01,-3.763870e-01
25%,-1.102492e-01,-7.762241e-03,-4.918644e-03,-1.408881e-02,-8.948642e-02,-7.735985e-03,-9.507567e-02,-2.702282e-02,-8.092618e-01,-1.166364e-02,...,-1.009507e+00,-9.543885e-01,-1.049659e+00,-4.390782e-01,-4.801968e-01,-2.891034e-01,-6.395319e-01,-6.248708e-01,-3.876346e-01,-3.763870e-01
50%,-1.102492e-01,-7.754745e-03,-4.918644e-03,-1.408881e-02,-8.948642e-02,-7.735985e-03,-9.507567e-02,-2.702282e-02,-8.092618e-01,-1.166364e-02,...,7.343426e-01,-4.756270e-01,-2.504011e-02,-3.332138e-01,-4.801968e-01,-2.891034e-01,-6.395319e-01,-6.248708e-01,-3.876346e-01,-3.763870e-01
75%,-1.102492e-01,-7.715224e-03,-4.790326e-03,-1.408881e-02,-8.948642e-02,-7.735985e-03,-9.507567e-02,-2.702282e-02,1.235694e+00,-1.166364e-02,...,7.343426e-01,1.258754e+00,1.066401e+00,-6.855302e-02,-2.860195e-01,-1.114257e-01,1.608759e+00,1.618955e+00,-3.876346e-01,-3.763870e-01
max,1.636428e+01,2.350675e+02,3.257486e+02,7.097831e+01,1.174348e+01,2.088191e+02,3.571955e+01,1.104972e+02,1.235694e+00,3.123689e+02,...,7.343426e-01,1.258754e+00,1.066401e+00,4.854138e+00,2.756092e+00,8.594782e+00,1.608759e+00,1.618955e+00,2.874410e+00,2.753914e+00


In [13]:
'label' in train_df.columns

True

In [14]:
train_df['label'].head(10)

0     normal
1     normal
2    neptune
3     normal
4     normal
5    neptune
6    neptune
7    neptune
8    neptune
9    neptune
Name: label, dtype: str

#### Step6 : Train/Test Feature-Target Split (X/y split)

In [15]:
X_train = train_df.drop(['label','binary_label','difficulty_level'],axis=1)
Y_train = train_df['binary_label']

X_test = test_df.drop(['label','binary_label','difficulty_level'],axis=1)
Y_test = test_df['binary_label']

In [16]:
print(X_train.shape)
print(Y_train.shape)

(125973, 122)
(125973,)


#### Step 7: Building the ANN Architecture

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [19]:
model= Sequential()
model.add(Dense(64,activation='relu',input_shape=(X_train.shape[1],)))
model.add(Dense(32,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

C:\Users\Suhitha\anaconda3\envs\sam\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │           7,872 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 9,985 (39.00 KB)

 Trainable params: 9,985 (39.00 KB)

 Non-trainable params: 0 (0.00 B)

#### Step 8: Compiling the Model

In [22]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

#### Step 9: Training the Model

In [26]:
history = model.fit(X_train, Y_train, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9863 - loss: 0.0423 - val_accuracy: 0.9919 - val_loss: 0.0235
Epoch 2/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9928 - loss: 0.0208 - val_accuracy: 0.9923 - val_loss: 0.0200
Epoch 3/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9940 - loss: 0.0175 - val_accuracy: 0.9935 - val_loss: 0.0188
Epoch 4/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9947 - loss: 0.0154 - val_accuracy: 0.9949 - val_loss: 0.0151
Epoch 5/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9950 - loss: 0.0139 - val_accuracy: 0.9954 - val_loss: 0.0140
Epoch 6/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9956 - loss: 0.0128 - val_accuracy: 0.9955 - val_loss: 0.0134
Epoch 7/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9960 - loss: 0.0119 - val_accuracy: 0.9957 - val_loss: 0.0138
Epoch 8/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9962 - loss: 0.0110 -

#### Step 10: Evaluation on the Test Set

In [27]:
from sklearn.metrics import classification_report,confusion_matrix

In [34]:
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

y_pred = model.predict(X_test)
y_pred_binary = (y_pred > 0.5).astype(int)

print(confusion_matrix(Y_test, y_pred_binary))
print(classification_report(Y_test, y_pred_binary))

705/705 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
[[8976  735]
 [4315 8518]]
              precision    recall  f1-score   support

           0       0.68      0.92      0.78      9711
           1       0.92      0.66      0.77     12833

    accuracy                           0.78     22544
   macro avg       0.80      0.79      0.78     22544
weighted avg       0.81      0.78      0.78     22544



#### Rebuilding the model with Dropout

In [35]:
from tensorflow.keras.layers import Dropout

model2 = Sequential()
model2.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model2.add(Dropout(0.3))
model2.add(Dense(32, activation='relu'))
model2.add(Dropout(0.3))
model2.add(Dense(1, activation='sigmoid'))

model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

C:\Users\Suhitha\anaconda3\envs\sam\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [37]:
history2 = model2.fit(X_train, Y_train, epochs=10, batch_size=32, validation_split=0.2)


Epoch 1/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.9785 - loss: 0.0688 - val_accuracy: 0.9915 - val_loss: 0.0249
Epoch 2/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9898 - loss: 0.0310 - val_accuracy: 0.9927 - val_loss: 0.0214
Epoch 3/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9915 - loss: 0.0251 - val_accuracy: 0.9947 - val_loss: 0.0180
Epoch 4/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9922 - loss: 0.0222 - val_accuracy: 0.9941 - val_loss: 0.0174
Epoch 5/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9927 - loss: 0.0222 - val_accuracy: 0.9954 - val_loss: 0.0156
Epoch 6/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9936 - loss: 0.0196 - val_accuracy: 0.9944 - val_loss: 0.0162
Epoch 7/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9936 - loss: 0.0186 - val_accuracy: 0.9961 - val_loss: 0.0133
Epoch 8/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9940 - loss: 0.018

In [38]:
y_pred2 = model2.predict(X_test)
y_pred_binary2 = (y_pred2 > 0.5).astype(int)

print(confusion_matrix(Y_test, y_pred_binary2))
print(classification_report(Y_test, y_pred_binary2))

705/705 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
[[9262  449]
 [3916 8917]]
              precision    recall  f1-score   support

           0       0.70      0.95      0.81      9711
           1       0.95      0.69      0.80     12833

    accuracy                           0.81     22544
   macro avg       0.83      0.82      0.81     22544
weighted avg       0.84      0.81      0.81     22544

